In [16]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline



In [17]:
words=open("names.txt",'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [18]:
len(words)

32033

In [19]:
chars=sorted(list(set(''.join(words))))
stoi={s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos ={i:s for s,i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [20]:
block_size=3
X,Y=[],[]
for w in words:
    #print(w)
    context=[0]*block_size
    for ch in w+'.':
        ix=stoi[ch]
        X.append(context)
        Y.append(ix)
        #print(''.join(itos[i] for i in context),'---->',itos[ix]) #标签三个，预测下一个
        context=context[1:]+[ix]

X=torch.tensor(X)
Y=torch.tensor(Y)


In [21]:
C=torch.randn((27,2))#每一个字符用一个2维向量表示
C[5]

tensor([0.6413, 1.7377])

In [22]:
emb=C[X]
emb.shape #(样本数，第几个字符，2维编码)

torch.Size([228146, 3, 2])

In [23]:
W1=torch.randn((6,100))
b1=torch.randn(100)

torch.cat([emb[:,0,:],emb[:,1,:],emb[:,2,:]],1).shape
''':  ：取所有 32 个样本
   0  ：取每个样本中的第 1 个字符
   :  ：取这个字符的全部 2 个 embedding 数值
'''
h=emb.view(32,6) @ W1 + b1
h

RuntimeError: shape '[32, 6]' is invalid for input of size 1368876

In [24]:
W2=torch.randn((100,27))
b2=torch.randn(27)
logits=h@ W2+b2

logits.shape

torch.Size([32, 27])

In [25]:
counts=logits.exp()
prob=counts/counts.sum(1,keepdims=True)

prob.shape


torch.Size([32, 27])

In [26]:
loss=-prob[torch.arange(32),Y].log().mean()
loss

IndexError: shape mismatch: indexing tensors could not be broadcast together with shapes [32], [228146]

In [27]:
# 初始化参数
C = torch.randn((27, 2), requires_grad=True)
W1 = torch.randn((6, 100), requires_grad=True)
b1 = torch.randn(100, requires_grad=True)
W2 = torch.randn((100, 27), requires_grad=True)
b2 = torch.randn(27, requires_grad=True)

# 把所有需要训练的参数放进列表
parameters = [C, W1, b1, W2, b2]

print(sum(p.nelement() for p in parameters))

3481


In [ ]:
for _ in range(10):

    #minibatch
    ix=torch.randint(0,X.shape[0],(32,))
    # forward pass
    emb=C[X[ix]] #(32,3,2)
    h=torch.tanh(emb.view(-1,6) @ W1 + b1)
    logits=h @ W2 +b2
    loss=F.cross_entropy(logits,Y[ix])
    print(loss.item())
    
    #backward pass
    for p in parameters:
        p.grad=None
    loss.backward()

    for p in parameters:
        p.data+=-0.1*p.grad


16.997032165527344
15.812030792236328
15.008551597595215
14.30596923828125
13.653451919555664
13.035258293151855
12.464962005615234
11.977581977844238
11.547469139099121
11.144730567932129
